# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a practical workflow for loading and exploring the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
**https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json**

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)

# Show main metadata information
md = dataset.metadata
print(f"Name: {md.name}\nDescription: {md.description}\nIdentifier: {md.identifier}")
print(f"Keywords: {md.keywords}")
print(f"DOI: {md.identifier}, Version: {md.version}")

## 2. Data Overview
Review available record sets, their fields, and their `@id` fields as defined by the Croissant schema.

In [ ]:
# Retrieve all record sets and their fields by @id
record_sets = dataset.metadata.record_sets
print(f"Number of Record Sets: {len(record_sets)}")
for rs in record_sets:
    print(f"Record Set: {rs['@id']}")
    field_ids = []
    field_names = []
    for field in rs['field']:
        field_ids.append(field['@id'])
        field_names.append(field.get('name', '(no name)'))
    print(f"  Fields: {field_names}")
    print(f"  Field @ids: {field_ids}\n")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.
We reference each entity by their `@id`. The code below demonstrates extracting all records from each record set, resulting in a pandas DataFrame for each.

In [ ]:
# List all record set @ids here
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

# Load all records into DataFrames by record set @id
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for Record Set '{record_set_id}'. Columns: {list(df.columns)}\n")
# For illustration, show the columns and first records of the main table
if record_set_ids:
    main_rs = record_set_ids[0]  # Usually the main record set is first
    print(f"First 5 records in '{main_rs}':")
    display(dataframes[main_rs].head(5))

## 4. Exploratory Data Analysis (EDA)
Let's explore and process numeric fields. We select a representative numeric field using its `@id` (from Section 2 above), filter records, normalize values, and perform grouping. **Replace the variables below with valid `@id`s and field names from your specific dataset, as revealed in the overview**.

In [ ]:
# Specify the record set and field @ids to explore
rs_id = record_set_ids[0]  # Use the main record set
df = dataframes[rs_id]

# List candidate numeric fields by their @id or column name
print("Available columns in DataFrame (examples):")
print(list(df.columns))

# Assume a numeric field 'Coverage' exists; in practice, use the actual column/field name or @id as discovered above
numeric_field = 'Coverage'  # Replace with real @id or column name
# If not present, select another numeric column from printed output above
if numeric_field not in df.columns:
    # Fallback to the first float/integer-like column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break

threshold = 10
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with '{numeric_field}' > {threshold} (n={len(filtered_df)}):")
display(filtered_df.head())

# Normalize the numeric field
normalized_col = f"{numeric_field}_normalized"
filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized '{numeric_field}' for filtered records (top 5):")
display(filtered_df[[numeric_field, normalized_col]].head())

# Grouping example: group by a categorical field such as 'Description' or any categorical field
group_field = 'Description'  # Replace as needed by inspecting your columns
if group_field in df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame("Mean").reset_index()
    print(f"Grouped data by '{group_field}':")
    display(grouped_df.head())

## 5. Visualization
We can visualize data distributions, e.g., for the normalized coverage or any other numeric variable. Below, we present a histogram and a boxplot for the selected numeric field after filtering.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not filtered_df.empty:
    plt.figure(figsize=(10, 4))
    plt.subplot(1,2,1)
    sns.histplot(filtered_df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field}' (> {threshold})")
    plt.xlabel(numeric_field)

    plt.subplot(1,2,2)
    sns.boxplot(x=filtered_df[numeric_field])
    plt.title(f"Boxplot of '{numeric_field}'")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load and inspect a Croissant-structured dataset using `mlcroissant` referencing entities via their `@id`s,
- Extract all records from each record set,
- Explore and process numeric fields (e.g., filtering, normalizing, grouping),
- Visualize key numeric data distributions.

This workflow can be extended for more advanced feature engineering, additional visualization, or application of machine learning models on mass spectrometry data or other FAIR-compliant datasets.